# Neural Network Volatility Model

This notebook is focused on training a multilayer perceptron nerual network to predict future 20-day realized volatility.

The model uses teh same selected features and time-based train/test split as all the other models so we can compare it fairly.

The neural network provides a different nonlinear modeling approach from the
linear, tree ensemble, and statistical time-series models already evaluated.

# Imports

In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Set Paths

In [2]:
FEATURES_PATH = Path("../../data/processed/features")
MODELING_PATH = Path("../../data/processed/modeling")
MODEL_OUTPUT_PATH = MODELING_PATH / "neural_network_mlp"

MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Output directory:", MODEL_OUTPUT_PATH)

Output directory: ..\..\data\processed\modeling\neural_network_mlp


# Load the Feature Dataset

In [3]:
df = pd.read_csv(
    FEATURES_PATH / "feature_engineered_dataset.csv",
    parse_dates=["Date"],
)

In [4]:
df = df.sort_values(['Date', 'ticker']).reset_index(drop=True)

## Shape and Info of Dataset

In [5]:
print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Ticker count:", df["ticker"].nunique())

Dataset shape: (41370, 30)
Date range: 2018-01-31 00:00:00 to 2025-12-01 00:00:00
Ticker count: 21


In [6]:
df.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,fed_funds_rate_pct,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-31,AAPL,39.138020,0.002755,0.0146,13.54,2.72,1.26,0,1.41,...,-0.001378,0.002755,7.589440e-06,0.006855,0.000087,0.011013,0.009462,40.695438,0.961730,0.022707
1,2018-01-31,AGG,84.439857,0.000833,0.0146,13.54,2.72,1.26,0,1.41,...,-0.000491,0.000833,6.946851e-07,0.001156,0.000002,0.001981,0.001411,84.804251,0.995703,0.002252
2,2018-01-31,AMZN,72.544502,0.009090,0.0146,13.54,2.72,1.26,0,1.41,...,0.010048,0.009090,8.263170e-05,0.011328,0.000193,0.003308,0.009819,65.750550,1.103329,0.023577
3,2018-01-31,CAT,136.724533,-0.005984,0.0146,13.54,2.72,1.26,0,1.41,...,0.002100,0.005984,3.580998e-05,0.009638,0.000147,0.014251,0.012264,139.401691,0.980795,0.026032
4,2018-01-31,GLD,127.650002,0.006703,0.0146,13.54,2.72,1.26,0,1.41,...,0.001005,0.006703,4.493635e-05,0.004544,0.000031,0.005561,0.005660,126.463501,1.009382,0.007289


# Load the Selected Features

In [7]:
SELECTED_FEATURES_PATH = FEATURES_PATH / "selected_features.csv"

In [8]:
NUMERIC_FEATURES = (
    pd.read_csv(SELECTED_FEATURES_PATH)["feature"]
    .dropna()
    .tolist()
)

In [9]:
missing_features = [
    feature for feature in NUMERIC_FEATURES
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        f"Selected features missing from dataset: {missing_features}"
    )

print("Selected numeric features:", len(NUMERIC_FEATURES))
NUMERIC_FEATURES

Selected numeric features: 17


['return_lag_1',
 'return_lag_5',
 'rolling_return_5d',
 'rolling_return_20d',
 'abs_return',
 'squared_return',
 'rolling_abs_return_20d',
 'rolling_volatility_5d',
 'price_to_moving_avg_20d',
 'vix',
 'treasury_10yr_pct',
 'yield_curve_spread',
 'is_inverted',
 'fed_funds_rate_pct',
 'unemployment_rate_pct',
 'recession_flag',
 'cpi_pct_change']

# Add Ticker Indicators
Ticker dummy variables let the neural network distinguish one asset from another.

In [10]:
ticker_dummies = pd.get_dummies(
    df["ticker"],
    prefix="ticker",
    dtype=float,
)

In [11]:
df = pd.concat([df, ticker_dummies], axis=1)


In [12]:
TICKER_FEATURES = ticker_dummies.columns.tolist()
FEATURE_COLUMNS = NUMERIC_FEATURES + TICKER_FEATURES
TARGET_COLUMN = "future_volatility_20d"

In [13]:
print("Numeric features:", len(NUMERIC_FEATURES))
print("Ticker features:", len(TICKER_FEATURES))
print("Total features:", len(FEATURE_COLUMNS))

Numeric features: 17
Ticker features: 21
Total features: 38


# Prepare the modeling data

In [14]:
model_df = df.dropna(
    subset=FEATURE_COLUMNS + [TARGET_COLUMN]
).copy()

In [16]:
model_df = model_df.sort_values("Date").reset_index(drop=True)

In [17]:
print("Rows before dropna:", len(df))
print("Rows after dropna:", len(model_df))
print("Missing feature values:", model_df[FEATURE_COLUMNS].isna().sum().sum())
print("Missing target values:", model_df[TARGET_COLUMN].isna().sum())

Rows before dropna: 41370
Rows after dropna: 41370
Missing feature values: 0
Missing target values: 0


# Create the time-based train/test split
Want to keep this date identical to the existing models.

In [19]:
SPLIT_DATE = pd.Timestamp("2024-01-01")

In [20]:
train_df = model_df[model_df["Date"] < SPLIT_DATE].copy()
test_df = model_df[model_df["Date"] >= SPLIT_DATE].copy()

In [21]:
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

In [22]:
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

In [23]:
split_metadata = {
    "model_run_timestamp": pd.Timestamp.now().isoformat(),
    "split_date": SPLIT_DATE.date().isoformat(),
    "train_start_date": train_df["Date"].min().date().isoformat(),
    "train_end_date": train_df["Date"].max().date().isoformat(),
    "test_start_date": test_df["Date"].min().date().isoformat(),
    "test_end_date": test_df["Date"].max().date().isoformat(),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
}

In [24]:
print(
    f"Train: {len(train_df):,} rows, "
    f"{train_df['Date'].min().date()} to "
    f"{train_df['Date'].max().date()}"
)

print(
    f"Test: {len(test_df):,} rows, "
    f"{test_df['Date'].min().date()} to "
    f"{test_df['Date'].max().date()}"
)


Train: 31,269 rows, 2018-01-31 to 2023-12-29
Test: 10,101 rows, 2024-01-02 to 2025-12-01


In [25]:
split_metadata

{'model_run_timestamp': '2026-07-29T15:33:00.786612',
 'split_date': '2024-01-01',
 'train_start_date': '2018-01-31',
 'train_end_date': '2023-12-29',
 'test_start_date': '2024-01-02',
 'test_end_date': '2025-12-01',
 'train_rows': 31269,
 'test_rows': 10101}

# Create the model
Both the features and target are standardized because neural networks generally train better when variables are on similar scales.

In [26]:
feature_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "mlp",
        MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.001,
            batch_size=256,
            learning_rate="adaptive",
            learning_rate_init=0.001,
            max_iter=500,
            shuffle=True,
            early_stopping=False,
            random_state=42,
            verbose=False,
        ),
    ),
])

In [27]:
mlp_model = TransformedTargetRegressor(
    regressor=feature_pipeline,
    transformer=StandardScaler(),
)

In [28]:
mlp_model

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('mlp', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too la